In [163]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

 
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.tree import export_text



In [164]:

data = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv"
!wget $data -O car_fuel_efficiency.csv

df = pd.read_csv('car_fuel_efficiency.csv')
df.head()

--2025-11-02 23:20:49--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 874188 (854K) [text/plain]
Saving to: ‘car_fuel_efficiency.csv’

car_fuel_efficiency 100%[===================>] 853.70K  --.-KB/s    in 0.04s   

2025-11-02 23:20:49 (23.2 MB/s) - ‘car_fuel_efficiency.csv’ saved [874188/874188]



,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


In [165]:
# fill missing values with 0
df.fillna(0, inplace=True)


In [166]:
df.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,0.0,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,0.0,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


In [167]:
# setup train, val, test splits
# df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
# df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
 
# df_train = df_train.reset_index(drop=True)
# df_val = df_val.reset_index(drop=True)
# df_test = df_test.reset_index(drop=True)

 
# y_train = df_train.fuel_efficiency_mpg.values
# y_val = df_val.fuel_efficiency_mpg.values
# y_test = df_test.fuel_efficiency_mpg.values
 
# del df_train['fuel_efficiency_mpg']
# del df_val['fuel_efficiency_mpg']
# del df_test['fuel_efficiency_mpg'] 

In [168]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
 
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
# Define your target column
target = 'fuel_efficiency_mpg'

# Separate features and target for each split
X_train = df_train.drop(columns=[target])
y_train = df_train[target]

X_val = df_val.drop(columns=[target])
y_val = df_val[target]

X_test = df_test.drop(columns=[target])
y_test = df_test[target]


In [169]:
# Identify numeric and categorical columns
numeric_features = ["engine_displacement", "num_cylinders", "horsepower",
                    "vehicle_weight", "acceleration", "model_year", "num_doors"]

categorical_features = ["origin", "fuel_type", "drivetrain"]

In [170]:
# remove categorial columns 
df_val = df_val.drop(columns=categorical_features)
df_train = df_train.drop(columns=categorical_features)
df_test = df_test.drop(columns=categorical_features)

In [171]:
df_train.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,num_doors,fuel_efficiency_mpg
0,120,5.0,169.0,2966.679505,13.9,2005,-1.0,15.301475
1,200,3.0,143.0,2950.822121,17.1,2013,-1.0,15.331215
2,180,6.0,180.0,3078.221669,17.4,2007,0.0,15.336679
3,280,5.0,174.0,2797.991793,0.0,2016,0.0,15.865850
4,250,4.0,133.0,2362.426930,16.3,2010,-1.0,18.102203


In [172]:
# one-hot encode categorical variables
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'  # Keep numeric columns as they are
)


In [173]:
# Question 1 Let's train a decision tree regressor to predict the fuel_efficiency_mpg variable.
# Which feature is used for splitting the data?

In [174]:

train_dicts = df_train.fillna(0).to_dict(orient='records')
train_dicts[:5]
dv = DictVectorizer(sparse=True)
dt = DecisionTreeRegressor(random_state=1)


# keep dv fitted 
X_train = dv.fit_transform(train_dicts)
dt.fit(X_train, y_train)

model = DecisionTreeRegressor(random_state=1)

model.fit(X_train, y_train)




,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,1
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [175]:
# print(export_text(dt))
names = dv.get_feature_names_out().tolist()
print(export_text(dt, feature_names=names))

|--- fuel_efficiency_mpg <= 14.96
|   |--- fuel_efficiency_mpg <= 12.47
|   |   |--- fuel_efficiency_mpg <= 10.63
|   |   |   |--- fuel_efficiency_mpg <= 9.24
|   |   |   |   |--- fuel_efficiency_mpg <= 8.16
|   |   |   |   |   |--- fuel_efficiency_mpg <= 7.37
|   |   |   |   |   |   |--- fuel_efficiency_mpg <= 6.50
|   |   |   |   |   |   |   |--- value: [6.20]
|   |   |   |   |   |   |--- fuel_efficiency_mpg >  6.50
|   |   |   |   |   |   |   |--- fuel_efficiency_mpg <= 6.98
|   |   |   |   |   |   |   |   |--- acceleration <= 9.45
|   |   |   |   |   |   |   |   |   |--- value: [6.81]
|   |   |   |   |   |   |   |   |--- acceleration >  9.45
|   |   |   |   |   |   |   |   |   |--- value: [6.89]
|   |   |   |   |   |   |   |--- fuel_efficiency_mpg >  6.98
|   |   |   |   |   |   |   |   |--- horsepower <= 135.50
|   |   |   |   |   |   |   |   |   |--- model_year <= 2013.50
|   |   |   |   |   |   |   |   |   |   |--- value: [7.19]
|   |   |   |   |   |   |   |   |   |--- model_yea

In [176]:
# Question 1 Let's train a decision tree regressor to predict the fuel_efficiency_mpg variable.
# Which feature is used for splitting the data?
# vehicle_weight

In [ ]:
# Question 2
# Train a random forest regressor with these parameters:

# n_estimators=10
# random_state=1
# n_jobs=-1 (optional - to make training faster)
# What's the RMSE of this model on the validation data?
# 9.36 4.5


In [178]:
model = RandomForestRegressor(n_estimators=10, random_state=1,n_jobs=-1)

model.fit(X_train, y_train)

# 2. Make predictions on the validation set
y_pred = model.predict(df_val)

# 3. Compute RMSE
mse = mean_squared_error(y_val, y_pred)

rmse = np.sqrt(mse)

print(f"RMSE on validation data: {rmse:.2f}")


RMSE on validation data: 9.36


/home/codespace/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
